# KNN Baseline With Sequential Feature Selection

This notebook trains a compact KNN regression baseline on the cleaned train/validation files. Sequential feature selection is run before final training so the model uses a smaller feature set. Hyperparameter optimization can come later.

In [5]:
import numpy as np
import pandas as pd

from sklearn.compose import TransformedTargetRegressor
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Keep these fixed so SFS results are reproducible.
RANDOM_STATE = 42
N_FEATURES_TO_SELECT = 25
SFS_SAMPLE_SIZE = 5000


In [6]:
# Load the pre-cleaned train/validation split from the data folder.
clean_train = pd.read_csv("data/clean_train.csv")
clean_val = pd.read_csv("data/clean_val.csv")

# Separate predictors from the PM2.5 target.
metadata_cols = ['Place_ID', 'Date', 'Place_ID X Date']
drop_cols = ['target'] + [col for col in metadata_cols if col in clean_train.columns]

X_train = clean_train.drop(columns=drop_cols)
y_train = clean_train["target"]
X_val = clean_val.drop(columns=drop_cols)
y_val = clean_val["target"]

print(X_train.shape, X_val.shape)


(24546, 70) (6011, 70)


## Sequential Feature Selection

SFS is run on a fixed sample to keep the notebook responsive. The final KNN model is then trained on the full cleaned training set using the selected features.

In [ ]:
# Run SFS on a sample to keep the notebook fast.
sfs_sample_size = min(SFS_SAMPLE_SIZE, len(X_train))
X_sfs = X_train.sample(n=sfs_sample_size, random_state=RANDOM_STATE)
y_sfs = y_train.loc[X_sfs.index]

# KNN needs scaling because it relies on distances between rows.
knn_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    KNeighborsRegressor(),
)

# Train on log target values, then convert predictions back to the original scale.
knn_for_selection = TransformedTargetRegressor(
    regressor=knn_pipeline,
    func=np.log1p,
    inverse_func=np.expm1,
)

cv = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

sfs = SequentialFeatureSelector(
    estimator=knn_for_selection,
    n_features_to_select=N_FEATURES_TO_SELECT,
    direction="forward",
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
)

sfs.fit(X_sfs, y_sfs)

selected_features = X_train.columns[sfs.get_support()].tolist()
selected_features


## Find Best Number Of Features

Run this cell when you want to compare several SFS feature counts. It can take a while because it fits a separate sequential selector for each value.

In [ ]:
# Try different feature counts and evaluate each selected set on validation data.
feature_count_results = []
feature_counts_to_try = [5, 8, 10, 12, 15, 20]

for n_features in feature_counts_to_try:
    print(f"Running SFS for {n_features} features...")

    sfs = SequentialFeatureSelector(
        estimator=knn_for_selection,
        n_features_to_select=n_features,
        direction="forward",
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1,
    )

    sfs.fit(X_sfs, y_sfs)
    features = X_train.columns[sfs.get_support()].tolist()

    model = TransformedTargetRegressor(
        regressor=make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            KNeighborsRegressor(),
        ),
        func=np.log1p,
        inverse_func=np.expm1,
    )

    model.fit(X_train[features], y_train)
    predictions = model.predict(X_val[features])

    feature_count_results.append({
        "n_features": n_features,
        "rmse": mean_squared_error(y_val, predictions, squared=False),
        "mae": mean_absolute_error(y_val, predictions),
        "r2": r2_score(y_val, predictions),
        "features": features,
    })

feature_count_results = (
    pd.DataFrame(feature_count_results)
    .sort_values("rmse")
    .reset_index(drop=True)
)

feature_count_results


## Save Best KNN Features

This saves the best 30-feature SFS result. The metrics describe the validation performance of the selected feature set as a whole, not each feature individually.

In [15]:
# Save the selected 30-feature set and its validation metrics.
best_30 = feature_count_results.loc[
    feature_count_results["n_features"].eq(30)
].iloc[0]

best_features = pd.DataFrame({
    "knn": best_30["features"],
    "validation_rmse_knn": best_30["rmse"],
})

best_features.to_csv("data/best_features.csv", index=False)
best_features


,knn,validation_rmse_knn
0,precipitable_water_entire_atmosphere,32.848392
1,relative_humidity_2m_above_ground,32.848392
2,specific_humidity_2m_above_ground,32.848392
3,temperature_2m_above_ground,32.848392
4,u_component_of_wind_10m_above_ground,32.848392
5,v_component_of_wind_10m_above_ground,32.848392
6,L3_NO2_NO2_column_number_density,32.848392
7,L3_NO2_NO2_slant_column_number_density,32.848392
8,L3_NO2_absorbing_aerosol_index,32.848392
9,L3_NO2_sensor_altitude,32.848392


## Train And Evaluate KNN

This uses the default KNN regressor settings for now. The target is transformed with `log1p` during training and predictions are converted back with `expm1`. We can tune neighbors, weights, and distance metric later.

In [ ]:
# Fit the final model on all cleaned training rows, using only SFS-selected columns.
knn_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    KNeighborsRegressor(),
)

knn_model = TransformedTargetRegressor(
    regressor=knn_pipeline,
    func=np.log1p,
    inverse_func=np.expm1,
)

knn_model.fit(X_train[selected_features], y_train)
val_pred = knn_model.predict(X_val[selected_features])

# Evaluate predictions after they have been converted back to PM2.5 units.
metrics = {
    "rmse": mean_squared_error(y_val, val_pred, squared=False),
    "mae": mean_absolute_error(y_val, val_pred),
    "r2": r2_score(y_val, val_pred),
}

metrics


{'rmse': 35.63798774192898,
 'mae': 23.65047062910318,
 'r2': 0.27594101988399666}